# 在 Kaggle 上用 COCO 2017 訓練 YOLO11x

這不是手機照片的預設流程。手機偵測的預設模型仍是 `yolo11m`。這本筆記本訓練最大的 **YOLO11x**。

**預設是微調**：參數格的 `MODE = "finetune"` 會載入官方預訓練的 `yolo11x.pt`。從零訓練要把同一格改成 `MODE = "scratch"`（`yolo11x.yaml`）。從零訓練非常久。Kaggle 每週 GPU 額度大約 **30 小時**，從零訓練要以週來算，不是一個 12 小時工作階段。估計表在下一節，那些數字是估計，不是實測。

說明書是同目錄的 `README.md`。不要在這個儲存庫裡登入 Kaggle，也不要在開發機上真的開 COCO 訓練。

## English

This notebook trains **YOLO11x** on COCO 2017. The phone-photo default stays `yolo11m`. **`MODE = "finetune"` is the default** and loads official pretrained `yolo11x.pt`. `MODE = "scratch"` trains `yolo11x.yaml` from random weights. Scratch training takes many weeks of Kaggle's weekly GPU quota of about **30 hours**. The table below is an estimate, not a measurement.


In [ ]:
# =============================================================================
# SWITCH. Default is fine-tuning the official pretrained checkpoint.
# "finetune" -> yolo11x.pt   (default, recommended)
# "scratch"  -> yolo11x.yaml from random weights (very long; see the quota table)
# =============================================================================
MODE = "finetune"

# "t4x2": two 16 GB T4s, device 0,1. "p100": one 16 GB P100, device 0.
GPU = "t4x2"

# Ultralytics patience. In Ultralytics 8.4, fitness is validation mAP50-95.
# 10 to 20 is the useful range. 0 disables this rule only.
PATIENCE = 15

# Custom stop. Total val loss is box + cls + dfl.
# A drop smaller than LOSS_MIN_DELTA for LOSS_PATIENCE epochs in a row stops.
# MAP_TARGET is validation mAP50-95. Official YOLO11x is about 0.547 on COCO val,
# so 0.55 means "near the published number". 0.50 would stop a pretrained
# fine-tune almost immediately. None disables the target.
LOSS_MIN_DELTA = 0.01
LOSS_PATIENCE = 15
MAP_TARGET = 0.55

# Stop this many hours into the 12-hour session so the last checkpoint can be saved.
TIME_HOURS = 11.0

# Extra epochN.pt files. 0 keeps only last.pt and best.pt. Each file is large.
SAVE_PERIOD = 10

IMGSZ = 640
WORKERS = 2  # 4 CPU cores and about 29 GB RAM. COCO decoding will OOM the CPU at 8.
AMP = True

# COCO does not fit in /kaggle/working (~20 GB saved). Scratch is wiped at session end.
# /kaggle/temp is not a documented separate disk; the next cell checks free space.
DATASET_DIR = "/kaggle/temp/coco"

# Attach your own Ultralytics-layout COCO and set this, or leave "" to download.
# No public Kaggle dataset slug is assumed.
COCO_INPUT = ""

# Continue a previous session. Example: "/kaggle/input/coco-yolo11x-ckpt/last.pt"
RESUME_LAST = ""

# Directory that contains src/detect/coco_early_stop.py (clone of this repo).
REPO_ROOT = "/kaggle/working/object-recognition"

PROJECT = "/kaggle/working/coco-yolo11x"

if MODE == "finetune":
    MODEL = "yolo11x.pt"
    PRETRAINED = True
    EPOCHS = 100
elif MODE == "scratch":
    MODEL = "yolo11x.yaml"
    PRETRAINED = False
    EPOCHS = 300
else:
    raise ValueError('MODE must be "finetune" (default) or "scratch"')

if GPU == "t4x2":
    # Two separate 16 GB cards. batch is the total and must be >= 2.
    # 8 means 4 images per GPU with AMP. Try 16 only if epoch 1 does not OOM.
    DEVICE = "0,1"
    BATCH = 8
elif GPU == "p100":
    DEVICE = "0"
    BATCH = 4  # one 16 GB P100, no tensor cores. Drop to 2 on OOM.
else:
    raise ValueError('GPU must be "t4x2" or "p100"')

# ESTIMATE, not a measurement. Hours use the rounded rates 1.1 and 4.5.
# Sessions use 11 useful hours inside a 12-hour limit. Weeks use a 30-hour quota.
ESTIMATE = {
    ("t4x2", "finetune"): {"hours_per_epoch": 1.1, "epochs": 100, "hours": 110, "sessions": 10, "weeks": 3.7},
    ("t4x2", "scratch"): {"hours_per_epoch": 1.1, "epochs": 300, "hours": 330, "sessions": 30, "weeks": 11.0},
    ("p100", "finetune"): {"hours_per_epoch": 4.5, "epochs": 100, "hours": 450, "sessions": 41, "weeks": 15.0},
    ("p100", "scratch"): {"hours_per_epoch": 4.5, "epochs": 300, "hours": 1350, "sessions": 123, "weeks": 45.0},
}
row = ESTIMATE[(GPU, MODE)]
print(
    f"MODE={MODE} model={MODEL} device={DEVICE} batch={BATCH} epochs={EPOCHS} "
    f"patience={PATIENCE} map_target={MAP_TARGET}"
)
print(
    "ESTIMATE (not measured): "
    f"{row['hours_per_epoch']} h/epoch, {row['hours']} h if the {row['epochs']}-epoch cap is used, "
    f"{row['sessions']} sessions, {row['weeks']} weeks of a 30-hour quota."
)
if MODE == "scratch":
    print(
        "From-scratch training is very long under the about-30-hour weekly quota: "
        f"about {row['weeks']} weeks ({row['hours']} hours, {row['sessions']} sessions) on this GPU."
    )
if EPOCHS != row["epochs"]:
    print("EPOCHS was edited. Scale hours, sessions, and weeks by the new epoch count.")


## 估計（不是實測）

沒有在 Kaggle 上量過。提早停止可以更短，所以總時數是「跑滿輪數上限」的估計。

假設：YOLO11x 前向 194.9 GFLOP（640，Ultralytics 模型表），反向按 2 倍，再乘 1.3 含 mosaic、讀圖和 5,000 張驗證。訓練集 118,287 張。一張 T4 假設只用到約 65 FP16 Tensor TFLOPS 的 20%。兩張 T4 假設 1.7 倍而不是 2 倍。P100 沒有 Tensor Core，假設用到約 18.7 FP16 TFLOPS 的 30%。四捨五入後 T4 x2 **每輪 1.1 小時**，P100 **每輪 4.5 小時**。利用率沒量過，上下 40% 都算合理。每個 12 小時工作階段按 11 小時可訓練來除。每週額度按 30 小時來除。

| 機器 | 模式 | 每輪 | 上限 | 總小時 | 工作階段 | 30 小時額度 |
| --- | --- | --- | --- | --- | --- | --- |
| T4 x2 | 微調 `yolo11x.pt`（預設） | 1.1 小時 | 100 | 110 | 10 | 約 3.7 週 |
| T4 x2 | 從零 `yolo11x.yaml` | 1.1 小時 | 300 | 330 | 30 | **11 週** |
| P100 | 微調（預設） | 4.5 小時 | 100 | 450 | 41 | 15 週 |
| P100 | 從零 | 4.5 小時 | 300 | 1350 | 123 | **45 週** |

從零訓練在 T4 x2 上大約 330 小時，也就是大約 11 週的額度、30 次工作階段。P100 上大約 1350 小時、45 週。預設保持 `finetune`。

Kaggle 文件仍列出 P100（1 GPU、4 CPU、約 29 GB）和 T4 x2（2 張 T4、4 CPU、約 29 GB）。有些 2026 的第二手文章說 P100 已下線。以 Accelerator 選單為準。兩張 T4 是兩張 16 GB 的卡。`batch=-1` 的 AutoBatch 只支援單卡，雙卡不要用。

### English

Not measured. Early stopping can finish sooner. The totals assume the epoch cap is used. Rounded rates are 1.1 h/epoch on T4 x2 and 4.5 h/epoch on P100, from the FLOP assumptions in `kaggle/README.md` (±40% is a fair band). Divide by 11 useful hours per 12-hour session and by a 30-hour weekly quota. Scratch training is about **330 hours, 30 sessions, 11 weeks** on T4 x2 and about **1,350 hours, 123 sessions, 45 weeks** on a P100.


## 磁碟與資料集

`/kaggle/working` 大約只能留下 **20 GB**。COCO 不要放這裡。`/kaggle/working` 以外的空間工作階段結束會被清掉。文件沒有保證有獨立的 `/kaggle/temp`；下一格會建立 `DATASET_DIR` 並檢查剩餘空間。

Ultralytics `coco.yaml` 會下載約 19 GB 的 `train2017.zip` 和約 1 GB 的 `val2017.zip`，而且目前的下載函式不解壓後刪 zip，所以尖峰大約 **40 GB**。剩餘空間少於約 45 GB 又沒有 `COCO_INPUT` 時，下一格會停下來。

`last.pt` 和 `best.pt` 含 optimizer，每個大約幾百 MB 到 1 GB，放得進 `/kaggle/working`。

沒有查證過某個公開 Kaggle COCO 代號。只有當你附加的資料集已經是 Ultralytics 版面（`images/train2017`）時才設定 `COCO_INPUT`。

### English

Do not put COCO in `/kaggle/working` (about 20 GB of saved output). Scratch outside that folder is wiped. `/kaggle/temp` is a convenient path, not a documented separate disk. Peak COCO use is about 40 GB because the zips are kept. The next cell refuses to download below about 45 GB free. No public dataset slug is assumed.


In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("ultralytics") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.3"])
print("ultralytics is installed. COCO is not downloaded in this cell.")


In [ ]:
import shutil
from pathlib import Path

import ultralytics
import yaml
from ultralytics import settings as ultra_settings

working = Path("/kaggle/working")
scratch = Path(DATASET_DIR)
if scratch == working or working in scratch.parents:
    raise RuntimeError("DATASET_DIR must sit outside /kaggle/working.")

scratch.parent.mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(scratch.parent).free / (1024**3)
print(f"Free space beside {scratch}: {free_gb:.1f} GiB")
if not COCO_INPUT and free_gb < 45:
    raise RuntimeError(
        f"Need about 45 GiB free for COCO zips plus extracted images, found {free_gb:.1f} GiB. "
        "Attach a COCO dataset and set COCO_INPUT, or choose a larger DATASET_DIR."
    )

working.mkdir(parents=True, exist_ok=True)
working_free = shutil.disk_usage(working).free / (1024**3)
if working_free < 5:
    raise RuntimeError(f"/kaggle/working has {working_free:.1f} GiB free. Checkpoints need a few GiB.")

dataset_root = Path(DATASET_DIR)
if COCO_INPUT:
    dataset_root = Path(COCO_INPUT)
    if not ((dataset_root / "images" / "train2017").is_dir() or (dataset_root / "train2017").is_dir()):
        nested = dataset_root / "coco"
        if (nested / "images" / "train2017").is_dir():
            dataset_root = nested
        else:
            raise FileNotFoundError(
                f"{COCO_INPUT} is not an Ultralytics COCO folder (expected images/train2017). "
                "No public Kaggle slug is assumed."
            )

# Absolute path so the download script writes outside /kaggle/working.
ultra_settings.update(datasets_dir=str(dataset_root.parent))
coco_yaml = Path(ultralytics.__file__).resolve().parent / "cfg" / "datasets" / "coco.yaml"
text = coco_yaml.read_text(encoding="utf-8").replace("path: coco", f"path: {dataset_root.as_posix()}", 1)
data_yaml = working / "coco_kaggle.yaml"
data_yaml.write_text(text, encoding="utf-8")
loaded = yaml.safe_load(text)
print(f"data yaml: {data_yaml}")
print(f"dataset path: {loaded['path']}")
print("Download happens on the training cell if images are missing. This cell does not download COCO.")


## 跨過 12 小時限制

1. 跑完後，`/kaggle/working/last.pt` 和 `/kaggle/working/best.pt` 都要在。權重目錄裡也有一份。
2. Save Version。下一個筆記本用 Add Input 選這份輸出，或先把輸出做成 Dataset。
3. 檔案在唯讀的 `/kaggle/input/<名稱>/`。把 `RESUME_LAST` 設成那個 `last.pt`。
4. 下一格會把 `last.pt` 複製到 `/kaggle/working`（輸入目錄不能寫），再用 `resume=True` 接著訓練。
5. COCO 在暫存碟上，下一輪要再下載或再附加。這次傳入的 `data` 會指向新路徑。

`patience`、`time`、`batch`、`device`、`workers`、`save_period` 續訓時可以改。其他超參數沿用 checkpoint。

### English

Save the notebook output as a Kaggle Dataset, attach it next session, set `RESUME_LAST` to the read-only `last.pt`, and continue with `resume=True`. The notebook copies that file into `/kaggle/working` first. COCO scratch is gone, so download or attach it again.


In [ ]:
import sys
from pathlib import Path

repo = Path(REPO_ROOT)
module = repo / "src" / "detect" / "coco_early_stop.py"
if not module.is_file():
    raise FileNotFoundError(
        "src/detect/coco_early_stop.py was not found at REPO_ROOT. "
        "Clone this repository into /kaggle/working/object-recognition "
        "(the revision that contains that file) and set REPO_ROOT. "
        "Do not put a Kaggle API token in the notebook."
    )
sys.path.insert(0, str(repo))
from src.detect.coco_early_stop import AcceptableResultStop, StopConfig

print(f"stop rules: {module}")


In [ ]:
import shutil
import time
from pathlib import Path

from ultralytics import YOLO
from ultralytics.utils import SETTINGS

# Skip the Ultralytics phone-home check. This notebook does not log into Kaggle.
SETTINGS["sync"] = False

resume_weights = None
if RESUME_LAST:
    source = Path(RESUME_LAST)
    if not source.is_file():
        raise FileNotFoundError(source)
    resume_weights = Path("/kaggle/working/coco-yolo11x/resume/last.pt")
    resume_weights.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, resume_weights)
    print(f"Copied {source} to {resume_weights}")

stopper = AcceptableResultStop(
    StopConfig(
        loss_min_delta=LOSS_MIN_DELTA,
        loss_patience=LOSS_PATIENCE,
        map_target=MAP_TARGET,
        time_budget_hours=TIME_HOURS,
    ),
    started_at=time.time(),
)

if resume_weights is not None:
    model = YOLO(str(resume_weights))
else:
    model = YOLO(MODEL)

model.add_callback("on_fit_epoch_end", stopper.on_fit_epoch_end)

save_dir = str(Path(PROJECT) / "train")
if resume_weights is not None:
    # resume=True reloads hyperparameters from last.pt, then applies the allowed overrides.
    model.train(
        resume=True,
        data=str(data_yaml),
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        workers=WORKERS,
        patience=PATIENCE,
        save_period=SAVE_PERIOD,
        time=TIME_HOURS,
        save_dir=save_dir,
        exist_ok=True,
    )
else:
    model.train(
        data=str(data_yaml),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        workers=WORKERS,
        patience=PATIENCE,
        amp=AMP,
        pretrained=PRETRAINED,
        project=PROJECT,
        name="train",
        exist_ok=True,
        save=True,
        save_period=SAVE_PERIOD,
        time=TIME_HOURS,
        cache=False,  # do not cache 118k COCO images in RAM or on the 20 GB disk
        plots=False,  # training plots are not required and use disk
        val=True,
        seed=0,
    )


In [ ]:
import shutil
from pathlib import Path

weights = Path(PROJECT) / "train" / "weights"
last = weights / "last.pt"
best = weights / "best.pt"
missing = [path for path in (last, best) if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Missing checkpoints: {missing}")

shutil.copy2(last, Path("/kaggle/working/last.pt"))
shutil.copy2(best, Path("/kaggle/working/best.pt"))
print(f"Kept {last} and {best}")
print("Also copied to /kaggle/working/last.pt and /kaggle/working/best.pt")
if stopper.last_decision is not None and stopper.last_decision.stop:
    print(f"Stopped early: {stopper.last_decision.reason}")


## 自動停止與多 GPU

`PATIENCE` 交給 Ultralytics。8.4 的 fitness 就是驗證 mAP50-95。

`on_fit_epoch_end` 另外在三種情況停下來：驗證損失平台、mAP50-95 達到 `MAP_TARGET`、或超過 `TIME_HOURS`。它在這一輪的驗證和存檔之後才設 `trainer.stop`，並把 `epochs` 收到剛剛跑完的那一輪，所以 `best.pt` 和 `last.pt` 會留下。

雙卡時只有 rank 0 做決定。Ultralytics 8.4 會在回呼之後把 `trainer.stop` 廣播到其他 rank，也會把回呼 pickle 進 DDP worker。若新版本不再廣播或 pickle 失敗，把 `DEVICE` 改成 `"0"` 只用一張卡，或只靠 `patience`。

### English

`patience` is Ultralytics fitness early stopping (mAP50-95 in 8.4). The callback also stops on a flat validation loss, on the mAP50-95 target, and on the time budget, and it does so only after that epoch has been validated and saved. On two GPUs, rank 0 sets `trainer.stop` and Ultralytics 8.4 broadcasts it. If that fails, use `device=0` or patience only.
